# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('/content/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A line chart fits change over time. Indexing to 1970 = 100 makes growth comparable across continents of different sizes. Africa is in orange and the rest are in grey, with direct labels in place of a legend. Africa grew 3.9×, the fastest of any continent.*

In [3]:
cont = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
               .sort_values(['Continent', 'Year']))
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first') * 100

colors = {c: '#BBBBBB' for c in cont['Continent'].unique()}
colors['Africa'] = '#E69F00'

fig = px.line(cont, x='Year', y='Index', color='Continent',
              color_discrete_map=colors, template='plotly_white',
              title="<b>Africa's population has nearly quadrupled since 1970 — far faster than any other continent</b>"
                    "<br><sup>Population indexed to 1970 = 100</sup>")

fig.update_traces(line=dict(width=2))
fig.update_traces(selector=dict(name='Africa'), line=dict(width=4))

nudge = {'Oceania': 10, 'Asia': -10}
end = cont[cont['Year'] == 2022]
for _, r in end.iterrows():
    is_af = r['Continent'] == 'Africa'
    fig.add_annotation(x=2022, y=r['Index'],
                       text=f"{r['Continent']} ({r['Index']/100:.1f}×)",
                       showarrow=False, xanchor='left', xshift=6,
                       yshift=nudge.get(r['Continent'], 0),
                       font=dict(color='#E69F00' if is_af else '#777777',
                                 size=13 if is_af else 11))

fig.add_hline(y=100, line_dash='dot', line_color='#999999', line_width=1)

fig.update_layout(showlegend=False, margin=dict(r=170), height=520)
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2020])
fig.update_yaxes(title=None, showgrid=True, gridcolor='#EEEEEE')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A sorted bar chart fits because the question ranks 15 countries by one number, and horizontal bars keep the country names readable. African countries are blue and the others are light grey, so Africa’s 4 countries (Nigeria, Ethiopia, DR Congo, Egypt) stand out. Value labels replace the x-axis.*

In [5]:
df['Added (M)'] = (df['2022 Population'] - df['1970 Population']) / 1_000_000
top15 = df.sort_values('Added (M)', ascending=False).head(15)

top15['Group'] = 'Other'
top15.loc[top15['Continent'] == 'Africa', 'Group'] = 'Africa'

fig = px.bar(top15, x='Added (M)', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa': '#0072B2', 'Other': 'lightgrey'},
             text_auto='.0f', template='plotly_white',
             title='<b>4 of the 15 countries that added the most people are in Africa</b><br><sup>Population added 1970–2022, in millions</sup>')

fig.update_yaxes(title=None, categoryorder='total ascending')
fig.update_xaxes(visible=False)
fig.update_traces(textposition='outside')
fig.update_layout(legend_title=None, height=550)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A scatter plot shows the relationship between two numerical variables. Log scales are needed because area and population range from tiny states to giants, and on linear axes most points would pile up in a corner. Colour by continent answers the “does it differ by continent” part using a palette safe for colour-blind readers with no red/green. Only the outliers are labelled, so the reader’s eye goes to the countries that break the pattern.*

In [6]:
outliers = ['India', 'China', 'Bangladesh', 'Singapore', 'Macau', 'Russia', 'Canada', 'Australia', 'Greenland', 'Mongolia']
df['Label'] = ''
df.loc[df['Country'].isin(outliers), 'Label'] = df['Country']

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', text='Label',
                 log_x=True, log_y=True, hover_name='Country', template='plotly_white',
                 color_discrete_sequence=['#E69F00', '#56B4E9', '#0072B2', '#CC79A7', '#332288', '#555555'],
                 title='<b>Bigger countries usually have more people — dense Asian countries sit above the trend, Greenland far below</b>'
                       '<br><sup>Land area vs 2022 population (log scales)</sup>')

fig.update_traces(textposition='top center', textfont_size=10, marker=dict(size=8, opacity=0.8))
fig.update_xaxes(title='Land area (km²)', showgrid=False)
fig.update_yaxes(title='Population 2022', showgrid=False)
fig.update_layout(legend_title=None, height=600)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A box plot shows the spread of density within each continent (median, quartiles, extremes), not just the average, and the dots show every individual country. A log scale is needed. The continents are sorted by median. Dark orange highlights South America, the least crowded continent, and the others are blue-grey context.*

In [11]:
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index

fig = px.box(df, x='Continent', y='Density', color='Continent', points='all',
             hover_name='Country', log_y=True, template='plotly_white',
             category_orders={'Continent': list(order)},
             color_discrete_map={'South America': '#E66101', 'Africa': '#B0B7C3', 'Asia': '#B0B7C3', 'Europe': '#B0B7C3',
                                 'North America': '#B0B7C3', 'Oceania': '#B0B7C3'},
             title='<b>Density varies far more within continents than between them — South America is the least crowded</b>'
                   '<br><sup>People per km² in 2022 (log scale), sorted by median</sup>')

fig.update_xaxes(title=None)
fig.update_yaxes(title=None, showgrid=False, dtick=1)
fig.update_layout(showlegend=False, height=550)
fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A choropleth map fits because the question asks where in the world countries grew or shrank. A diverging purple–orange scale centred at 0% separates growth (purple) from decline (orange). The colour range is capped at −25% to +100% so extreme cases like Qatar don’t wash out every other country. The frame and coastlines are removed to declutter the map.*

In [12]:
df['Change (%)'] = (df['2022 Population'] - df['2000 Population']) / df['2000 Population'] * 100

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change (%)',
                    hover_name='Country', template='plotly_white',
                    color_continuous_scale='PuOr', color_continuous_midpoint=0, range_color=[-25, 100],
                    title='<b>Populations shrank across Eastern Europe and Japan, while Africa and the Gulf grew fastest</b>'
                          '<br><sup>Population change 2000–2022 (%) — orange = shrank, purple = grew</sup>')

fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(coloraxis_colorbar=dict(title=None, ticksuffix='%'), height=550, margin=dict(l=0, r=0, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A treemap shows a part-to-whole split across two levels in one figure: continents, then the countries inside them. Box area equals population, so Asia’s dominance and the size of India and China are obvious at a glance. Colour encodes continent, using a palette safe for colour-blind readers (blue, orange, magenta, mustard, sky blue, grey) with no red/green. Each box is labelled with its share of world population, so no legend is needed.*

In [13]:

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', template='plotly_white',
                 color_discrete_map={'(?)': 'white', 'Asia': '#4477AA', 'Africa': '#EE7733', 'Europe': '#AA3377',
                                     'North America': '#CCBB44', 'South America': '#66CCEE', 'Oceania': '#BBBBBB'},
                 title='<b>Almost 6 in 10 people live in Asia — India and China alone hold over a third of the world</b>'
                       '<br><sup>2022 population by continent and country (box size = population)</sup>')

fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}', root_color='white')
fig.update_layout(height=650, margin=dict(l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*A 100% stacked bar chart shows how the shares of a whole change over time. Every bar adds up to 100%. Africa sits at the bottom of the stack on a flat baseline, which makes its growth easy to read.*

In [14]:
df_long['Region'] = 'Rest of the world'
df_long.loc[df_long['Continent'].isin(['Africa', 'Asia', 'Europe']), 'Region'] = df_long['Continent']

share = df_long.groupby(['Year', 'Region'], as_index=False)['Population'].sum()
share['Share (%)'] = share['Population'] / share.groupby('Year')['Population'].transform('sum') * 100

fig = px.bar(share, x='Year', y='Share (%)', color='Region', text_auto='.0f', template='plotly_white',
             category_orders={'Region': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
             color_discrete_map={'Africa': '#D6604D', 'Asia': '#4393C3', 'Europe': '#92C5DE', 'Rest of the world': '#E0E0E0'},
             title="<b>Africa's share of the world doubled since 1970, while Europe's halved</b>"
                   '<br><sup>Share of world population (%)</sup>')

fig.update_xaxes(title=None, type='category')
fig.update_yaxes(visible=False)
fig.update_layout(legend_title=None, bargap=0.2, height=550)
fig.show()